<a href="https://colab.research.google.com/github/avindumihisara0229-code/ErgoSense/blob/Tharusha/KKT_Deshapriya_DSGP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Feature extraction**

In [1]:
!pip install librosa soundfile


In [2]:
import os
import numpy as np
import pandas as pd
import librosa


In [3]:
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/archive/audio_speech_actors_01-24"


In [4]:
def get_stress_label(filename):
    """
    Extract stress label from RAVDESS filename
    Stressed = Angry (05), Fearful (06)
    Not stressed = Neutral (01), Calm (02), Happy (03)
    """
    emotion_code = int(filename.split("-")[2])

    if emotion_code in [5, 6]:
        return 1  # Stressed
    elif emotion_code in [1, 2, 3]:
        return 0  # Not stressed
    else:
        return None  # Ignore other emotions


In [5]:
def extract_features(file_path):
    # Load audio (use only first 3 seconds for consistency)
    y, sr = librosa.load(file_path, duration=3, offset=0.5)

    # MFCCs (13 coefficients)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs, axis=1)

    # Pitch (fundamental frequency)
    pitches, _ = librosa.piptrack(y=y, sr=sr)
    pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

    # Energy (RMS)
    energy = np.mean(librosa.feature.rms(y=y))

    # Tempo (speech rate)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    # Combine all features into one vector
    return np.hstack([mfccs_mean, pitch_mean, energy, tempo])


In [7]:
features = []
labels = []

for actor_folder in os.listdir(DATASET_PATH):
    actor_path = os.path.join(DATASET_PATH, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                label = get_stress_label(file)

                if label is not None:
                    file_path = os.path.join(actor_path, file)
                    feature_vector = extract_features(file_path)

                    features.append(feature_vector)
                    labels.append(label)


In [8]:
feature_columns = [f"mfcc_{i}" for i in range(1, 14)]
feature_columns += ["pitch", "energy", "tempo"]

df = pd.DataFrame(features, columns=feature_columns)
df["stress"] = labels

df.head()


,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13,pitch,energy,tempo,stress
0,-584.674011,63.550861,3.216504,12.510523,6.313325,0.937045,-17.175444,-8.413464,-6.622798,-10.023497,-1.402356,-3.117262,-5.184851,1548.881470,0.005284,67.999589,0
1,-531.203979,61.647114,-4.572315,2.740763,-2.088609,3.958043,-19.650618,-11.051280,-7.474490,-12.864723,2.031493,-10.209840,-4.990941,1490.485962,0.007827,184.570312,0
2,-515.324646,85.207718,-8.088744,9.528167,1.951334,-6.190319,-21.997009,-5.105417,-3.040505,-14.841343,12.985037,-9.392869,-5.627865,1536.403687,0.005880,89.102909,0
3,-592.081482,77.136101,4.449088,18.031532,10.912557,0.447543,-10.532049,-5.788989,-0.668430,-6.633837,-0.983075,-0.901634,-1.960368,1299.168213,0.004334,123.046875,0
4,-579.352417,77.645363,6.332064,13.562618,7.578548,-1.311473,-15.640059,-5.363317,-2.596069,-8.506312,4.690220,-6.763521,-3.690127,1483.194580,0.003516,112.347147,0


In [9]:
df["stress"].value_counts()


,count
stress,
0,480
1,384


In [12]:
df.to_csv("ravdess_stress_features.csv", index=False)
print("Feature extraction complete. CSV saved!")


Feature extraction complete. CSV saved!


In [13]:
from google.colab import files
files.download("ravdess_stress_features.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Feature scaling + train/test split**


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [15]:
X = df.drop("stress", axis=1)
y = df["stress"]


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [17]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [19]:
print("Training data mean (approx 0):")
print(X_train_scaled.mean(axis=0)[:5])

print("\nTraining data std (approx 1):")
print(X_train_scaled.std(axis=0)[:5])


Training data mean (approx 0):
[-3.23908772e-16 -1.07969591e-16 -3.59898636e-17 -5.14140909e-17
 -1.02828182e-17]

Training data std (approx 1):
[1. 1. 1. 1. 1.]


**Handle class imbalance**

In [20]:
print("Class distribution:")
print(y_train.value_counts())


Class distribution:
stress
0    384
1    307
Name: count, dtype: int64


In [23]:
##Install imbalanced-learn
!pip install imbalanced-learn

In [24]:
##Import SMOTE
from imblearn.over_sampling import SMOTE

In [26]:
##Apply SMOTE (ONLY on training data)
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled, y_train
)


In [27]:
##Verify balanced classes
print("After SMOTE:")
print(pd.Series(y_train_smote).value_counts())


After SMOTE:
stress
1    384
0    384
Name: count, dtype: int64
